# Random-Walk Embeddings with Skip-Gram (No PyG Node2Vec)

We’ll
1. Sample a random graph from a constant graphon.
2. Split edges for link-prediction (train/val/test).
3. Generate random-walk sequences.
4. Train a pure-PyTorch Skip-Gram model (`SkipGramRW`) on walk pairs + negative sampling.
5. Plot training loss.
6. Benchmark embeddings by link-prediction AUC.

In [ ]:
# 0. Imports & Utils
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Data
from torch_geometric.utils import train_test_split_edges
from logic.graphon_generator import generate_constant_graphon, interp_graphon, sample_graph
from data.random_walk_dataset import RandomWalkDataset
from model.models import SkipGramRW
from sklearn.metrics import roc_auc_score

# fix random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)


## 1. Build Graph & Split Edges
- **Graph**: G(n=100, p=0.05) from a constant graphon
- **Edge split**: 80% train, 10% val, 10% test for link prediction

In [ ]:
# Parameters
num_nodes  = 100
graph_p    = 0.05

# 1.a) Sample graph
W          = generate_constant_graphon(p=graph_p, resolution=200)
graph_fn   = interp_graphon(W)
G_nx       = sample_graph(graph_fn, num_nodes)

# 1.b) Build PyG Data and split edges
edge_index = torch.tensor(list(G_nx.edges()), dtype=torch.long).t().contiguous()
# undirected => both directions
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
data       = Data(num_nodes=num_nodes, edge_index=edge_index)
data       = train_test_split_edges(data, val_ratio=0.1, test_ratio=0.1)

print(f"Total nodes: {data.num_nodes}")
print(f"Train edges: {data.train_pos_edge_index.size(1)}")
print(f"Val pos/neg: {data.val_pos_edge_index.size(1)}/{data.val_neg_edge_index.size(1)}")
print(f"Test pos/neg: {data.test_pos_edge_index.size(1)}/{data.test_neg_edge_index.size(1)}")

## 2. Visualize Sample Random Walks
Dataset returns sequences of node indices of length `walk_length+1`.

In [ ]:
walk_length     = 10
walks_per_node  = 5

rw_ds = RandomWalkDataset(
    n_nodes= num_nodes,
    walk_length= walk_length,
    graphon_fn= graph_fn
)

for i in range(5):
    walk = rw_ds[i].numpy()
    print(f"Walk {i:>2d}:", walk)

## 3. Prepare Skip-Gram Training
- **Model input**: `(centers, contexts, negatives)` batches from walks.
- **Embedding dim**: 64, **negatives/node**: 5.
- **Loss**: Noise‐Contrastive Estimation (NCE).

In [ ]:
# Hyperparameters
embedding_dim  = 64
context_size   = 2   # window each side
neg_samples    = 5
batch_size     = 128
epochs         = 5

# Model & optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = SkipGramRW(num_nodes, embedding_dim).to(device)
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)

# DataLoader wraps full walks
loader = DataLoader(rw_ds, batch_size=batch_size, shuffle=True)

loss_history = []

### 3.a) Training Loop
For each batch of walks:
1. Slide a window to extract (center, context) pairs.  
2. Sample `neg_samples` negatives per pair.  
3. Compute NCE loss and backprop.

In [ ]:
for epoch in range(1, epochs+1):
    model.train()
    total_loss = 0.0
    total_pairs = 0

    for walks in loader:  # walks: (B, L+1)
        walks = walks.to(device)
        centers, contexts = [], []
        # build positive pairs
        for path in walks:
            for i in range(context_size, walk_length + 1 - context_size):
                c_idx = path[i].item()
                for offset in range(-context_size, context_size+1):
                    if offset == 0:
                        continue
                    centers.append(c_idx)
                    contexts.append(path[i + offset].item())

        centers  = torch.tensor(centers,  dtype=torch.long, device=device)
        contexts = torch.tensor(contexts, dtype=torch.long, device=device)
        total_pairs += centers.size(0)

        # negatives: uniform random
        negatives = torch.randint(
            0, num_nodes,
            (centers.size(0), neg_samples),
            device=device
        )

        # forward + backward
        loss = model(centers, contexts, negatives)
        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item() * centers.size(0)

    avg_loss = total_loss / total_pairs
    loss_history.append(avg_loss)
    print(f"Epoch {epoch}/{epochs} — Loss: {avg_loss:.4f}")

## 4. Training Loss Curve

In [ ]:
plt.figure(figsize=(5,4))
plt.plot(loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Avg NCE Loss')
plt.title('Skip-Gram Training Loss')
plt.grid(True)
plt.show()

## 5. Link-Prediction Benchmark
- We use the learned **target embeddings** (from `model.get_embeddings()`).
- Score edges by dot-product of embeddings: higher = more likely.
- Evaluate on **validation** edges (val_pos vs val_neg) with ROC-AUC.

In [ ]:
model.eval()
with torch.no_grad():
    Z = model.get_embeddings().cpu().numpy()  # (N, D)

def score_edges(edge_idx):
    u, v = edge_idx
    return np.sum(Z[u] * Z[v], axis=1)

pos_val = data.val_pos_edge_index.cpu().numpy()
neg_val = data.val_neg_edge_index.cpu().numpy()

scores  = np.concatenate([score_edges(pos_val), score_edges(neg_val)])
labels  = np.concatenate([np.ones(pos_val.shape[1]), np.zeros(neg_val.shape[1])])
auc_val = roc_auc_score(labels, scores)
print(f"Validation ROC-AUC: {auc_val:.4f}")